# Cross-topic Argument Mining with RoBERTa (Colab)

Re-implements Stab et al. (EMNLP 2018), *Cross-topic Argument Mining from Heterogeneous Sources*, with **RoBERTa replacing the BiCLSTM**.

- UKP corpus is read as a single CSV from Google Drive.
- DIP2016 (folder of XML files) is used as the auxiliary task for the MTL variant.
- All hyperparameters and seeds are gathered in one config cell below.
- Per-run JSONs + a `summary.json` are written to your chosen output folder on Drive.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install dependencies

In [ ]:
!pip install -q "transformers>=4.41" "datasets>=2.19" "scikit-learn>=1.3" "pandas>=2.0" "tqdm>=4.66"

## 3. Get the source code

Either clone the repo, or upload the `src/` folder yourself. Adjust the URL/branch as needed.

In [ ]:
import os, sys
REPO_DIR = '/content/ACC'
REPO_URL = 'https://github.com/ali26sami/acc.git'
BRANCH = 'claude/replace-biclstm-roberta-TCffX'

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch && git checkout {BRANCH} && git pull

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('Repo ready at', REPO_DIR)

## 4. Configuration  (edit me)

Every knob — paths, hyperparameters, seeds, which setups to run — lives here.

In [ ]:
from pathlib import Path

# ----- Paths on Google Drive -----
DRIVE_ROOT     = Path('/content/drive/MyDrive')
UKP_CSV        = DRIVE_ROOT / 'UKP' / 'ukp_sentential_argument_mining.csv'   # <-- set to your actual file
DIP_DIR        = DRIVE_ROOT / 'DIP2016'                                      # folder of 50 XML files
OUTPUT_DIR     = DRIVE_ROOT / 'roberta_argmining_runs'                       # results land here
QUERY_TEXT_CSV = None   # optional: CSV with columns [queryID, query_text] to give DIP a real topic string

# ----- Model / tokenization -----
MODEL_NAME = 'roberta-base'
MAX_LENGTH = 128

# ----- Optimization -----
EPOCHS           = 10
BATCH_SIZE       = 32
EVAL_BATCH_SIZE  = 64
LR               = 2e-5
WEIGHT_DECAY     = 0.01
WARMUP_RATIO     = 0.06
GRAD_CLIP        = 1.0

# ----- Experimental protocol -----
SEEDS         = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]   # paper uses 10 seeds
LABEL_SETUPS  = [2, 3]                            # 2-label and 3-label
MTL_MODES     = [False, True]                     # False = single-task; True = MTL+DIP2016
TEST_TOPICS   = None  # None = all 8 topics from the paper; or e.g. ['gun control']

# ----- MTL specifics -----
DIP_MAX_EXAMPLES = 300_000   # paper: "use 300K of 600K". Set None to use all.

# ----- Misc -----
NUM_WORKERS = 2

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output dir:', OUTPUT_DIR)
assert UKP_CSV.exists(), f'UKP file not found: {UKP_CSV}'
assert DIP_DIR.exists(), f'DIP folder not found: {DIP_DIR}'


### 4b. Optional: queryID → query text map for DIP2016

The XML files only carry `queryID`. Provide a CSV with columns `queryID,query_text` if you want DIP samples to use a real query string as the topic (closer to the paper). If you skip this, the queryID number is used as the topic text.

In [ ]:
import pandas as pd

QUERY_TEXT_MAP: dict[str, str] = {}
if QUERY_TEXT_CSV is not None and Path(QUERY_TEXT_CSV).exists():
    qdf = pd.read_csv(QUERY_TEXT_CSV)
    QUERY_TEXT_MAP = {str(q): str(t) for q, t in zip(qdf['queryID'], qdf['query_text'])}
    print('Loaded', len(QUERY_TEXT_MAP), 'queryID -> text mappings')
else:
    print('No queryID->text map provided; will use queryID as text.')

## 5. Sanity-check the data

In [ ]:
from src.data import load_ukp_csv, load_dip2016, TOPICS

df = load_ukp_csv(UKP_CSV)
print('UKP rows:', len(df))
print('UKP topics:', sorted(df['topic_norm'].unique()))
print('UKP label counts:')
print(df.groupby(['topic_norm', 'set', 'annotation']).size().unstack(fill_value=0).head(20))

dip = load_dip2016(DIP_DIR, query_text_map=QUERY_TEXT_MAP or None, limit_files=3)
print('\nDIP sample (first 3 files):', len(dip), 'sentences')
for ex in dip[:3]:
    print(f'  query={ex.query[:40]!r} label={ex.label} sent={ex.sentence[:80]!r}')

## 6. Smoke test — 1 topic, 1 seed, single-task

Confirms everything is wired before launching the full grid.

In [ ]:
from src.train import HParams, run_one

hp_smoke = HParams(
    ukp_csv=UKP_CSV, dip_dir=DIP_DIR, output_dir=OUTPUT_DIR,
    model_name=MODEL_NAME, max_length=MAX_LENGTH,
    epochs=2, batch_size=BATCH_SIZE, eval_batch_size=EVAL_BATCH_SIZE,
    lr=LR, weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO,
    grad_clip=GRAD_CLIP,
    test_topic='gun control', num_labels=2, seed=0,
    use_mtl=False, num_workers=NUM_WORKERS,
)
smoke = run_one(hp_smoke)
print(smoke)

## 7. Full run (all topics × all seeds × all setups)

Each completed run is written to `OUTPUT_DIR/<tag>.json`. Already-completed runs are skipped, so the cell is resumable.

In [ ]:
import json
from dataclasses import replace
from src.data import TOPICS
from src.metrics import aggregate_runs
from src.train import HParams, run_one

base_hp = HParams(
    ukp_csv=UKP_CSV, dip_dir=DIP_DIR, output_dir=OUTPUT_DIR,
    model_name=MODEL_NAME, max_length=MAX_LENGTH,
    epochs=EPOCHS, batch_size=BATCH_SIZE, eval_batch_size=EVAL_BATCH_SIZE,
    lr=LR, weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO,
    grad_clip=GRAD_CLIP, num_workers=NUM_WORKERS,
    dip_max_examples=DIP_MAX_EXAMPLES,
    dip_query_text_map=QUERY_TEXT_MAP,
)

topics_to_run = TEST_TOPICS or TOPICS
summary: dict = {}

for use_mtl in MTL_MODES:
    for num_labels in LABEL_SETUPS:
        per_topic: dict[str, list[dict]] = {}
        for topic in topics_to_run:
            runs: list[dict] = []
            for seed in SEEDS:
                mtl_tag = 'mtl' if use_mtl else 'single'
                tag = f"{mtl_tag}_L{num_labels}_{topic.replace(' ', '_')}_seed{seed}"
                out_path = OUTPUT_DIR / f'{tag}.json'
                if out_path.exists():
                    runs.append(json.loads(out_path.read_text()))
                    print(f'skip (cached): {tag}')
                    continue
                hp = replace(
                    base_hp,
                    test_topic=topic, num_labels=num_labels,
                    seed=seed, use_mtl=use_mtl,
                )
                print(f'==> {tag}')
                res = run_one(hp)
                out_path.write_text(json.dumps(res, indent=2))
                runs.append(res)
            per_topic[topic] = runs
        key = f"{'mtl' if use_mtl else 'single'}_{num_labels}label"
        summary[key] = {t: aggregate_runs(r) for t, r in per_topic.items()}
        all_runs = [r for rs in per_topic.values() for r in rs]
        summary[key]['__overall__'] = aggregate_runs(all_runs)

(OUTPUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2))
print('Saved summary to', OUTPUT_DIR / 'summary.json')

## 8. Inspect results

In [ ]:
import json
from pathlib import Path

summary = json.loads((OUTPUT_DIR / 'summary.json').read_text())
for setup, by_topic in summary.items():
    print(f'\n=== {setup} ===')
    overall = by_topic['__overall__']
    for k, (m, s) in overall.items():
        print(f'  {k:<14s} {m:.4f} ± {s:.4f}')